# 03 - Residual Stream Interventions

Runs the core experimental conditions:
- **self_prefill**: Re-tokenize COT, fresh forward pass, sample answer (sanity check)
- **zeroed_layer_k**: Replace residual at last position with raw embedding at layer k

**IMPORTANT**: Restart the kernel before running this notebook to free GPU memory from vLLM (notebook 02).

**Set `CONDITION` in the cell below** to control which condition runs.

**Fully resumable** - caches one JSON file per (condition, problem) in `cache/interventions/`.

In [17]:
import subprocess, sys
from pathlib import Path

WORKSPACE = Path("/workspace/10-4-2026")
REPO_DIR = WORKSPACE / "legibility"

if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)
else:
    WORKSPACE.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", "https://github.com/JackHopkins/legibility.git", str(REPO_DIR)], check=True)

sys.path.insert(0, str(REPO_DIR))
from lib.config import *

for d in [CACHE_DIR, COT_CACHE, INTERVENTION_CACHE, RESULTS_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

Updating cd62b45..d4f22f3
Fast-forward
 03_interventions.ipynb | 361 ++++++++++---------------------------------------
 1 file changed, 72 insertions(+), 289 deletions(-)


From https://github.com/JackHopkins/legibility
   cd62b45..d4f22f3  main       -> origin/main


In [18]:
# Verify GPU is free and purge errored cache files
import torch, json

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info()
    print(f"GPU memory: {free/1e9:.1f} GB free / {total/1e9:.1f} GB total")
    if free < 50e9:
        print("WARNING: Less than 50GB free. Restart the kernel to free vLLM memory.")

# Purge cached files that contain errors (so they get retried)
purged = 0
for p in INTERVENTION_CACHE.glob("*.json"):
    try:
        entry = json.loads(p.read_text())
        if entry.get("error") is not None:
            p.unlink()
            purged += 1
    except Exception:
        p.unlink()
        purged += 1

print(f"Purged {purged} errored cache files")

CUDA available: True
GPU memory: 128.8 GB free / 150.0 GB total
Purged 765 errored cache files


In [19]:
# =============================================
# SET THE CONDITION TO RUN HERE
# =============================================
# Options:
#   "self_prefill"
#   "zeroed_layer_0"
#   "zeroed_layer_9"
#   "zeroed_layer_18"
#   "zeroed_layer_27"
#   "zeroed_layer_35"
#
# Or set to "all" to run all conditions sequentially.

CONDITION = "all"

In [20]:
import json
import traceback
import torch
from tqdm.auto import tqdm
from lib.data import build_prefill_string
from lib.intervention import load_model, forward_pass_logits, extract_logit_stats

In [21]:
# Load model
model, tokenizer = load_model(MODEL_NAME, use_nnsight=True)

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Loaded Qwen/Qwen3-4B with nnsight


In [22]:
# Load COT results (from notebook 02)
cots_path = RESULTS_DIR / "cots.jsonl"
assert cots_path.exists(), f"Run 02_generate_cots.ipynb first. Missing: {cots_path}"

cots = []
with open(cots_path) as f:
    for line in f:
        entry = json.loads(line)
        if entry["error"] is None and entry["cot_text"] is not None:
            cots.append(entry)

print(f"Loaded {len(cots)} valid COT results")

Loaded 128 valid COT results


In [23]:
def parse_condition(condition_name: str) -> int | None:
    """Parse condition name to get the layer index for zeroing.
    Returns None for self_prefill (no intervention).
    """
    if condition_name == "self_prefill":
        return None
    elif condition_name.startswith("zeroed_layer_"):
        return int(condition_name.split("_")[-1])
    else:
        raise ValueError(f"Unknown condition: {condition_name}")


def run_condition(condition_name: str, cots: list[dict]):
    """Run a single experimental condition across all COT results."""
    zero_at_layer = parse_condition(condition_name)
    
    # Resume logic: strip "{condition_name}_" prefix from filename to get problem_id
    prefix = f"{condition_name}_"
    done_ids = set()
    for p in INTERVENTION_CACHE.glob(f"{prefix}*.json"):
        pid_str = p.stem[len(prefix):]
        done_ids.add(int(pid_str))

    todo = [c for c in cots if c["problem_id"] not in done_ids]
    print(f"\n[{condition_name}] Resuming: {len(done_ids)} done, {len(todo)} remaining")
    
    if not todo:
        return
    
    correct = 0
    total = 0
    
    for entry in tqdm(todo, desc=condition_name):
        try:
            prefill = build_prefill_string(entry["question"], entry["cot_text"])
            logits = forward_pass_logits(prefill, zero_at_layer=zero_at_layer)
            stats = extract_logit_stats(logits, entry["gold_answer"])
            
            cache_entry = {
                "problem_id": entry["problem_id"],
                "question": entry["question"],
                "gold_answer": entry["gold_answer"],
                "condition": condition_name,
                "cot_text": entry["cot_text"],
                "predicted_answer": stats["predicted_answer"],
                "top1_token": stats["top1_token"],
                "top1_prob": stats["top1_prob"],
                "gold_token_rank": stats["gold_token_rank"],
                "logits_top10": stats["logits_top10"],
                "error": None,
            }
            
            if stats["predicted_answer"] == entry["gold_answer"]:
                correct += 1
            total += 1
            
        except Exception:
            cache_entry = {
                "problem_id": entry["problem_id"],
                "question": entry["question"],
                "gold_answer": entry["gold_answer"],
                "condition": condition_name,
                "cot_text": entry["cot_text"],
                "predicted_answer": None,
                "top1_token": None,
                "top1_prob": None,
                "gold_token_rank": None,
                "logits_top10": None,
                "error": traceback.format_exc(),
            }
        
        fname = f"{prefix}{entry['problem_id']}.json"
        (INTERVENTION_CACHE / fname).write_text(json.dumps(cache_entry))
    
    if total > 0:
        print(f"[{condition_name}] Batch accuracy: {correct}/{total} = {correct/total:.1%}")

In [24]:
# Run the selected condition(s)
if CONDITION == "all":
    conditions_to_run = CONDITIONS
else:
    conditions_to_run = [CONDITION]

print(f"Conditions to run: {conditions_to_run}")

for cond in conditions_to_run:
    run_condition(cond, cots)

Conditions to run: ['self_prefill', 'zeroed_layer_0', 'zeroed_layer_9', 'zeroed_layer_18', 'zeroed_layer_27', 'zeroed_layer_35']

[self_prefill] Resuming: 2 done, 126 remaining


self_prefill:   0%|          | 0/126 [00:00<?, ?it/s]

[self_prefill] Batch accuracy: 0/126 = 0.0%

[zeroed_layer_0] Resuming: 1 done, 127 remaining


zeroed_layer_0:   0%|          | 0/127 [00:00<?, ?it/s]

[zeroed_layer_0] Batch accuracy: 0/127 = 0.0%

[zeroed_layer_9] Resuming: 0 done, 128 remaining


...

[zeroed_layer_18] Batch accuracy: 1/128 = 0.8%

[zeroed_layer_27] Resuming: 0 done, 128 remaining


zeroed_layer_27:   0%|          | 0/128 [00:00<?, ?it/s]

[zeroed_layer_27] Batch accuracy: 0/128 = 0.0%

[zeroed_layer_35] Resuming: 0 done, 128 remaining


zeroed_layer_35:   0%|          | 0/128 [00:00<?, ?it/s]

[zeroed_layer_35] Batch accuracy: 0/128 = 0.0%


In [25]:
# Aggregate results per condition into results/*.jsonl
for cond in CONDITIONS:
    prefix = f"{cond}_"
    entries = []
    for p in sorted(INTERVENTION_CACHE.glob(f"{prefix}*.json"), key=lambda x: int(x.stem[len(prefix):])):
        entries.append(json.loads(p.read_text()))
    
    if not entries:
        continue
    
    output_path = RESULTS_DIR / f"{cond}.jsonl"
    with open(output_path, "w") as f:
        for entry in entries:
            f.write(json.dumps(entry) + "\n")
    
    valid = [e for e in entries if e["error"] is None]
    correct = sum(1 for e in valid if e["predicted_answer"] == e["gold_answer"])
    errors = len(entries) - len(valid)
    
    if valid:
        print(f"{cond}: {correct}/{len(valid)} correct ({correct/len(valid):.1%}), {errors} errors -> {output_path}")
    else:
        print(f"{cond}: 0 valid, {errors} errors -> {output_path}")

self_prefill: 0/128 correct (0.0%), 0 errors -> /workspace/10-4-2026/results/self_prefill.jsonl
zeroed_layer_0: 0/128 correct (0.0%), 0 errors -> /workspace/10-4-2026/results/zeroed_layer_0.jsonl
zeroed_layer_9: 12/128 correct (9.4%), 0 errors -> /workspace/10-4-2026/results/zeroed_layer_9.jsonl
zeroed_layer_18: 1/128 correct (0.8%), 0 errors -> /workspace/10-4-2026/results/zeroed_layer_18.jsonl
zeroed_layer_27: 0/128 correct (0.0%), 0 errors -> /workspace/10-4-2026/results/zeroed_layer_27.jsonl
zeroed_layer_35: 0/128 correct (0.0%), 0 errors -> /workspace/10-4-2026/results/zeroed_layer_35.jsonl


In [26]:
# Quick summary of all conditions
print("\n" + "="*60)
print("SUMMARY")
print("="*60)

# Include normal (from COTs)
cots_valid = [c for c in cots if c["error"] is None]
normal_correct = sum(1 for c in cots_valid if c["predicted_answer"] == c["gold_answer"])
print(f"{'normal':<25} {normal_correct}/{len(cots_valid)} = {normal_correct/len(cots_valid):.1%}")

for cond in CONDITIONS:
    result_path = RESULTS_DIR / f"{cond}.jsonl"
    if not result_path.exists():
        print(f"{cond:<25} not yet run")
        continue
    
    entries = [json.loads(l) for l in result_path.read_text().splitlines()]
    valid = [e for e in entries if e["error"] is None]
    correct = sum(1 for e in valid if e["predicted_answer"] == e["gold_answer"])
    print(f"{cond:<25} {correct}/{len(valid)} = {correct/len(valid):.1%}")


SUMMARY
normal                    99/128 = 77.3%
self_prefill              0/128 = 0.0%
zeroed_layer_0            0/128 = 0.0%
zeroed_layer_9            12/128 = 9.4%
zeroed_layer_18           1/128 = 0.8%
zeroed_layer_27           0/128 = 0.0%
zeroed_layer_35           0/128 = 0.0%


In [ ]:
# Diagnostic: inspect self_prefill predictions vs gold
import json

sp_path = RESULTS_DIR / "self_prefill.jsonl"
cots_path = RESULTS_DIR / "cots.jsonl"

sp_entries = [json.loads(l) for l in sp_path.read_text().splitlines()]
cot_entries = {e["problem_id"]: e for e in [json.loads(l) for l in cots_path.read_text().splitlines()]}

print("Self-prefill: first 10 predictions vs gold")
print("-" * 70)
for e in sp_entries[:10]:
    cot = cot_entries.get(e["problem_id"], {})
    print(f"PID {e['problem_id']:>3} | gold={e['gold_answer']:<8} | predicted={str(e['predicted_answer']):<8} | top1='{e['top1_token']}' (p={e['top1_prob']})")
    # Show the end of the COT text
    if cot.get("cot_text"):
        print(f"         COT ends with: ...{cot['cot_text'][-80:]}")
    if cot.get("answer_portion"):
        print(f"         Answer portion: {cot['answer_portion'][:100]}")
    print()